In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [2]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Total Tours

In [3]:
def total_tours(data1, data2, tag='PSRC Region'):
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = Tour_1_total
    tpp[f'{survey_year}Survey'] = Tour_2_total
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.1f}',
        f'{survey_year}Survey': '{:,.1f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [4]:
total_tours(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,"5,878,074.0","5,050,722.0","827,351.9",16.4%


In [5]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
total_tours(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,"462,441.0","332,686.9","129,754.1",39.0%


## Tour per Person

In [6]:
def tour_per_person(data1, data2, tag='PSRC Region'):
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    ##Tours per person
    tpp1 = Tour_1_total / Person_1_total
    tpp2 = Tour_2_total / Person_2_total
    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = tpp1
    tpp[f'{survey_year}Survey'] = tpp2
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.1f}',
        f'{survey_year}Survey': '{:,.1f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [7]:
tour_per_person(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,1.4,1.3,0.0,2.2%


In [8]:
tour_per_person(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,1.4,1.1,0.3,23.9%


## Tours per Person by Person Type

In [9]:
from collections import OrderedDict

def tour_by_pptyp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose and Person Type/Number of Stops
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    PersonsDay1 = pd.merge(data1['Person'][['hhno', 'pno', 'pptyp', 'psexpfac']], data1['PersonDay'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    PersonsDay2 = pd.merge(data2['Person'][['hhno', 'pno', 'pptyp', 'psexpfac']], data2['PersonDay_cloned'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    # calculate the percentage of each number of stops by purpose
    data1['Tour']['all_stops'] = data1['Tour']['tripsh1'] + data1['Tour']['tripsh2'] - 2  # minus the origin and destination
    data2['Tour_cloned']['all_stops'] = data2['Tour_cloned']['tripsh1'] + data2['Tour_cloned']['tripsh2'] - 2  # minus the origin and destination
    intermediate_stopsbypurp1 = data1['Tour'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    intermediate_stopsbypurp2 = data2['Tour_cloned'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    # retain only the first 5 stops
    intermediate_stopsbypurp1 = intermediate_stopsbypurp1[intermediate_stopsbypurp1['all_stops']<=5]
    intermediate_stopsbypurp2 = intermediate_stopsbypurp2[intermediate_stopsbypurp2['all_stops']<=5]

    for purpose in data1['Tour']['pdpurp'].value_counts().index:
        if purpose == 'Work':
            tc = 'wktours'
            sc = 'wkstops'
        elif purpose == 'Social':
            tc = 'sotours'
            sc = 'sostops'
        elif purpose == 'School':
            tc = 'sctours'
            sc = 'scstops'
        elif purpose == 'Escort':
            tc = 'estours'
            sc = 'esstops'
        elif purpose == 'Personal Business':
            tc = 'pbtours'
            sc = 'pbstops'
        elif purpose == 'Shop':
            tc = 'shtours'
            sc = 'shstops'
        elif purpose == 'Meal':
            tc = 'mltours'
            sc = 'mlstops'
        #Merge a column to PersonsDay for the current purpose
        PersonsDay1 = PersonsDay1.merge(data1['PersonDay'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        PersonsDay2 = PersonsDay2.merge(data2['PersonDay_cloned'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        toursPersPurp1 = weighted_average(PersonsDay1, tc, 'psexpfac', 'pptyp')
        toursPersPurp2 = weighted_average(PersonsDay2, tc, 'psexpfac', 'pptyp')
        #Delete added column to make future iterations faster
        PersonsDay1.drop(columns = [tc], inplace = True)
        PersonsDay2.drop(columns = [tc], inplace = True)
        items = OrderedDict(((name1, toursPersPurp1), (name2, toursPersPurp2)))
        toursPersPurp = pd.DataFrame.from_dict(items)
        toursPersPurp = get_differences(toursPersPurp, name1, name2, 2)
        toursPersPurp = recode_index(toursPersPurp, 'pptyp','Person Type')
        toursPersPurp = toursPersPurp.loc[ptype_cat.values(), :]
        # display table
        display(toursPersPurp.style.format({
            name1: '{:,.2f}',
            name2: '{:,.2f}',
            f'Difference ({name1} - {name2})': '{:,.2f}',
            f'% Difference ({name1} - {name2})': '{:,.2f}%',
        }))
        # bar plot
        fig = px.bar(
            toursPersPurp.reset_index(),
            x='Person Type',
            y=[name1, name2],
            barmode='group',
            title=f'{purpose} Tours by Person Type'
        )
        fig.update_layout(yaxis_title=f'{purpose} Tours per Person', xaxis_title='Person Type', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
        fig.show()

        #Number of stops by purpose
        ps = pd.DataFrame()
        # fill missing 'all_stops' values with 0 for both intermediate_stopsbypurp1 and intermediate_stopsbypurp2
        imstpbypurp1 = intermediate_stopsbypurp1[intermediate_stopsbypurp1['pdpurp']==purpose].copy(deep=True)
        imstpbypurp2 = intermediate_stopsbypurp2[intermediate_stopsbypurp2['pdpurp']==purpose].copy(deep=True)
        # calculate percentage
        imstpbypurp1['percentage'] = imstpbypurp1['toexpfac'] / imstpbypurp1['toexpfac'].sum() * 100
        imstpbypurp2['percentage'] = imstpbypurp2['toexpfac'] / imstpbypurp2['toexpfac'].sum() * 100
        imstpbypurp1 = imstpbypurp1[imstpbypurp1['all_stops']<=5]
        imstpbypurp2 = imstpbypurp2[imstpbypurp2['all_stops']<=5]
        imstpbypurp1 = imstpbypurp1.set_index('all_stops').reindex(range(0, 6), fill_value=0).reset_index()
        imstpbypurp2 = imstpbypurp2.set_index('all_stops').reindex(range(0, 6), fill_value=0).reset_index()
        ps['% of Tours (' + name1 + ')'] = list(imstpbypurp1['percentage'])
        ps['% of Tours (' + name2 + ')'] = list(imstpbypurp2['percentage'])
        ps[purpose + ' Tours'] = range(0, 6)
        ps = ps.set_index(purpose + ' Tours')
        ps = get_differences( ps, '% of Tours (' + name1 + ')', '% of Tours (' + name2 + ')', 2)
        # table
        display(ps.style.format({
            f'% of Tours ({name1})': '{:,.1f}',
            f'% of Tours ({name2})': '{:,.1f}',
            f'Difference (% of Tours ({name1}) - % of Tours ({name2}))': '{:,.1f}',
            f'% Difference (% of Tours ({name1}) - % of Tours ({name2}))': '{:,.1f}%'
        }))

        # figure
        fig_stops = px.bar(
            ps.reset_index(),
            x=ps.index.name,
            y=[f'% of Tours ({name1})', f'% of Tours ({name2})'],
            barmode='group',
            title=f'Number of Stops per Tour by Purpose ({purpose}, {tag})'
        )
        fig_stops.update_layout(yaxis_title='Percentage of Tours', xaxis_title='Number of Stops', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
        fig_stops.update_yaxes(ticksuffix='%')
        fig_stops.show()

In [10]:
tour_by_pptyp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.65,0.66,-0.01,-1.89%
Part-Time Worker,0.61,0.52,0.09,17.35%
Non-Working Adult Age 65+,0.00,0.02,-0.02,-100.00%
Non-Working Adult Age <65,0.00,0.00,-0.00,-100.00%
University Student,0.38,0.21,0.17,78.79%
High School Student Age 16+,0.14,0.09,0.05,59.78%
Child Age 5-15,0.00,0.00,0.00,"5,528.26%"
Child Age 0-4,0.00,0.00,0.00,nan%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Work Tours,,,,
0,36.1,53.2,-17.1,-32.1%
1,28.2,22.2,6.0,26.8%
2,19.2,14.4,4.8,33.3%
3,9.8,7.4,2.5,33.2%
4,4.7,2.2,2.4,110.4%
5,2.0,0.5,1.5,271.8%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.32,0.23,0.09,36.61%
Part-Time Worker,0.36,0.31,0.05,15.39%
Non-Working Adult Age 65+,0.50,0.42,0.08,19.92%
Non-Working Adult Age <65,0.50,0.36,0.14,40.30%
University Student,0.22,0.32,-0.10,-31.96%
High School Student Age 16+,0.38,0.36,0.02,4.14%
Child Age 5-15,0.21,0.17,0.04,25.28%
Child Age 0-4,0.26,0.21,0.05,23.08%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Social Tours,,,,
0,75.1,66.0,9.1,13.8%
1,17.7,19.2,-1.5,-8.0%
2,5.4,7.9,-2.5,-31.5%
3,1.4,3.5,-2.1,-60.5%
4,0.4,1.9,-1.5,-81.2%
5,0.1,1.5,-1.4,-95.0%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.11,0.09,0.02,24.43%
Part-Time Worker,0.19,0.11,0.08,77.44%
Non-Working Adult Age 65+,0.30,0.25,0.05,18.95%
Non-Working Adult Age <65,0.43,0.27,0.16,58.29%
University Student,0.06,0.09,-0.03,-33.58%
High School Student Age 16+,0.06,0.06,-0.00,-3.15%
Child Age 5-15,0.11,0.03,0.08,254.74%
Child Age 0-4,0.27,0.09,0.18,189.71%


...

,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
School Tours,,,,
0,55.9,69.3,-13.4,-19.4%
1,23.7,24.9,-1.2,-4.9%
2,12.9,3.8,9.2,244.2%
3,5.2,1.3,3.9,300.2%
4,1.8,0.7,1.1,163.2%
5,0.6,0.1,0.5,694.3%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.10,0.10,-0.00,-4.67%
Part-Time Worker,0.10,0.12,-0.01,-9.07%
Non-Working Adult Age 65+,0.09,0.09,0.00,3.95%
Non-Working Adult Age <65,0.15,0.07,0.08,107.35%
University Student,0.05,0.03,0.01,41.67%
High School Student Age 16+,0.10,0.06,0.04,77.77%
Child Age 5-15,0.01,0.00,0.01,985.41%
Child Age 0-4,0.09,0.05,0.04,86.03%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Meal Tours,,,,
0,63.8,58.0,5.8,9.9%
1,23.2,22.8,0.4,1.7%
2,9.0,14.2,-5.2,-36.7%
3,2.9,3.1,-0.1,-4.9%
4,0.9,1.8,-1.0,-52.8%
5,0.2,0.1,0.2,275.1%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.06,0.05,0.01,29.56%
Part-Time Worker,0.13,0.10,0.03,31.24%
Non-Working Adult Age 65+,0.25,0.15,0.10,71.34%
Non-Working Adult Age <65,0.22,0.14,0.08,55.73%
University Student,0.03,0.14,-0.11,-77.76%
High School Student Age 16+,0.03,0.03,0.00,3.30%
Child Age 5-15,0.02,0.02,-0.00,-8.67%
Child Age 0-4,0.06,0.06,0.00,7.33%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Personal Business Tours,,,,
0,60.8,53.2,7.5,14.2%
1,25.5,28.9,-3.4,-11.9%
2,10.5,8.1,2.4,29.2%
3,2.5,4.8,-2.2,-46.9%
4,0.6,4.4,-3.8,-86.4%
5,0.1,0.5,-0.4,-77.7%


In [11]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
tour_by_pptyp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.38,0.22,0.17,76.43%
Part-Time Worker,0.43,0.36,0.07,19.15%
Non-Working Adult Age 65+,0.56,0.25,0.32,126.82%
Non-Working Adult Age <65,0.57,0.71,-0.14,-19.40%
University Student,0.27,0.44,-0.18,-40.16%
High School Student Age 16+,0.44,0.58,-0.13,-23.22%
Child Age 5-15,0.24,0.20,0.04,21.22%
Child Age 0-4,0.31,0.09,0.22,253.06%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Social Tours,,,,
0,74.6,59.7,14.9,25.0%
1,18.0,28.4,-10.4,-36.6%
2,5.5,4.7,0.8,16.2%
3,1.4,1.1,0.3,29.9%
4,0.4,3.1,-2.7,-87.6%
5,0.1,3.0,-2.9,-97.4%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.62,0.63,-0.01,-1.62%
Part-Time Worker,0.55,0.21,0.34,160.42%
Non-Working Adult Age 65+,0.00,0.00,-0.00,-100.00%
Non-Working Adult Age <65,0.00,0.00,0.00,nan%
University Student,0.32,0.04,0.28,669.74%
High School Student Age 16+,0.14,0.08,0.06,71.82%
Child Age 5-15,0.00,0.00,0.00,nan%
Child Age 0-4,0.00,0.00,0.00,nan%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Work Tours,,,,
0,37.4,46.9,-9.5,-20.3%
1,27.3,27.8,-0.5,-1.7%
2,18.8,21.1,-2.3,-10.8%
3,9.8,2.6,7.2,269.1%
4,4.7,1.6,3.1,193.7%
5,2.0,0.0,2.0,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.00,0.00,0.00,nan%
Part-Time Worker,0.00,0.00,-0.00,-100.00%
Non-Working Adult Age 65+,0.00,0.00,0.00,nan%
Non-Working Adult Age <65,0.00,0.00,-0.00,-100.00%
University Student,0.63,0.26,0.37,141.84%
High School Student Age 16+,0.74,0.25,0.49,198.75%
Child Age 5-15,0.74,0.83,-0.10,-11.59%
Child Age 0-4,0.00,0.27,-0.27,-100.00%


...

,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Shop Tours,,,,
0,52.1,67.2,-15.1,-22.4%
1,25.3,20.8,4.5,21.7%
2,14.7,8.3,6.4,76.6%
3,5.3,1.7,3.6,210.3%
4,1.9,1.6,0.4,23.0%
5,0.6,0.4,0.2,52.1%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.12,0.15,-0.03,-19.24%
Part-Time Worker,0.14,0.34,-0.20,-59.05%
Non-Working Adult Age 65+,0.12,0.03,0.08,255.49%
Non-Working Adult Age <65,0.18,0.01,0.17,"1,842.37%"
University Student,0.06,0.04,0.03,73.76%
High School Student Age 16+,0.13,0.05,0.08,147.27%
Child Age 5-15,0.02,0.00,0.01,"5,256.62%"
Child Age 0-4,0.11,0.00,0.11,nan%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Meal Tours,,,,
0,63.1,86.4,-23.2,-26.9%
1,24.1,8.7,15.3,176.0%
2,8.9,4.0,4.9,122.5%
3,2.8,0.9,1.9,210.3%
4,0.9,0.0,0.9,nan%
5,0.2,0.0,0.2,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.07,0.03,0.04,143.07%
Part-Time Worker,0.15,0.06,0.09,161.86%
Non-Working Adult Age 65+,0.24,0.17,0.07,41.71%
Non-Working Adult Age <65,0.20,0.06,0.14,244.24%
University Student,0.03,0.01,0.02,273.69%
High School Student Age 16+,0.03,0.05,-0.03,-50.46%
Child Age 5-15,0.02,0.00,0.02,"1,404.12%"
Child Age 0-4,0.06,0.00,0.06,"1,803.51%"


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Personal Business Tours,,,,
0,60.2,63.2,-3.0,-4.7%
1,25.6,32.3,-6.7,-20.7%
2,10.7,2.4,8.3,340.8%
3,2.7,2.1,0.6,30.0%
4,0.6,0.0,0.6,nan%
5,0.1,0.0,0.1,nan%


## Tours per Person by Purpose

In [12]:
def tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_1_total
    tpbp2 = data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Tours per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Tours per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Tours per Person (DaysimOutputs)', 'Tours per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'pdpurp', 'Tour Purpose')
    tpbp = tpbp.loc[pdpurp_cat.values()]
    # table
    display(tpbp.style.format({
        'Tours per Person (DaysimOutputs)': '{:,.1f}',
        f'Tours per Person ({survey_year}Survey)': '{:,.1f}',
        f"Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}',
        f"% Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Tour Purpose',
        y=['Tours per Person (DaysimOutputs)', f'Tours per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Tours per Person by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Tours per Person', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()    

In [13]:
tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Tours per Person (DaysimOutputs),Tours per Person (2023Survey),Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey)),% Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey))
Tour Purpose,,,,
Work,0.4,0.4,0.0,0.0%
School,0.1,0.1,0.0,3.9%
Escort,0.1,0.1,0.0,5.5%
Personal Business,0.1,0.1,0.0,4.9%
Shop,0.2,0.2,0.0,2.3%
Meal,0.1,0.1,0.0,5.7%
Social,0.3,0.3,0.0,0.9%


In [14]:
tours_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Tours per Person (DaysimOutputs),Tours per Person (2023Survey),Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey)),% Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey))
Tour Purpose,,,,
Work,0.3,0.3,0.0,11.5%
School,0.2,0.1,0.0,11.1%
Escort,0.1,0.1,0.0,16.3%
Personal Business,0.1,0.1,0.0,88.4%
Shop,0.1,0.1,0.1,61.2%
Meal,0.1,0.1,0.1,59.7%
Social,0.4,0.3,0.1,16.4%


## Tours per Person by Mode

In [15]:
from collections import OrderedDict

def tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose and Person Type/Number of Stops
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    PersonsDay1 = pd.merge(data1['Tour'][['hhno', 'pno', 'tmodetp']], data1['PersonDay'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    PersonsDay2 = pd.merge(data2['Tour_cloned'][['hhno', 'pno', 'tmodetp']], data2['PersonDay_cloned'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    # calculate the percentage of each number of stops by purpose
    data1['Tour']['all_stops'] = data1['Tour']['tripsh1'] + data1['Tour']['tripsh2'] - 2  # minus the origin and destination
    data2['Tour_cloned']['all_stops'] = data2['Tour_cloned']['tripsh1'] + data2['Tour_cloned']['tripsh2'] - 2  # minus the origin and destination
    intermediate_stopsbypurp1 = data1['Tour'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    intermediate_stopsbypurp2 = data2['Tour_cloned'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    # retain only the first 5 stops
    intermediate_stopsbypurp1 = intermediate_stopsbypurp1[intermediate_stopsbypurp1['all_stops']<=5]
    intermediate_stopsbypurp2 = intermediate_stopsbypurp2[intermediate_stopsbypurp2['all_stops']<=5]

    for purpose in data1['Tour']['pdpurp'].value_counts().index:
        if purpose == 'Work':
            tc = 'wktours'
            sc = 'wkstops'
        elif purpose == 'Social':
            tc = 'sotours'
            sc = 'sostops'
        elif purpose == 'School':
            tc = 'sctours'
            sc = 'scstops'
        elif purpose == 'Escort':
            tc = 'estours'
            sc = 'esstops'
        elif purpose == 'Personal Business':
            tc = 'pbtours'
            sc = 'pbstops'
        elif purpose == 'Shop':
            tc = 'shtours'
            sc = 'shstops'
        elif purpose == 'Meal':
            tc = 'mltours'
            sc = 'mlstops'
        #Merge a column to PersonsDay for the current purpose
        PersonsDay1 = PersonsDay1.merge(data1['PersonDay'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        PersonsDay2 = PersonsDay2.merge(data2['PersonDay_cloned'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        toursPersPurp1 = weighted_average(PersonsDay1, tc, 'pdexpfac', 'tmodetp')
        toursPersPurp2 = weighted_average(PersonsDay2, tc, 'pdexpfac', 'tmodetp')
        #Delete added column to make future iterations faster
        PersonsDay1.drop(columns = [tc], inplace = True)
        PersonsDay2.drop(columns = [tc], inplace = True)
        items = OrderedDict(((name1, toursPersPurp1), (name2, toursPersPurp2)))
        toursPersPurp = pd.DataFrame.from_dict(items)
        toursPersPurp = get_differences(toursPersPurp, name1, name2, 2)
        toursPersPurp = recode_index(toursPersPurp, 'tmodetp','Mode')
        # Retain only rows where the index (mode) is not 'Other'
        toursPersPurp = toursPersPurp[toursPersPurp.index != 'Other']
        toursPersPurp = toursPersPurp.loc[mode_cat.values(), :]
        # display table
        display(toursPersPurp.style.format({
            name1: '{:,.1f}',
            name2: '{:,.1f}',
            f'Difference ({name1} - {name2})': '{:,.1f}',
            f'% Difference ({name1} - {name2})': '{:,.1f}%',
        }))
        # bar plot
        fig = px.bar(
            toursPersPurp.reset_index(),
            x='Mode',
            y=[name1, name2],
            barmode='group',
            title=f'{purpose} Tours by Mode'
        )
        fig.update_layout(yaxis_title=f'{purpose} Tours per Person', xaxis_title='Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
        fig.show()

In [16]:
tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.4,0.5,-0.0,-9.1%
Bike,0.2,0.5,-0.3,-61.7%
SOV,0.6,0.6,0.0,0.8%
HOV2,0.3,0.3,-0.0,-10.6%
HOV3+,0.3,0.2,0.1,22.9%
Transit Walk Access,0.5,0.3,0.2,48.1%
Transit Auto Access,1.1,0.6,0.5,80.3%
School Bus,0.0,0.0,0.0,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.8,0.9,-0.2,-19.1%
Bike,0.7,0.5,0.1,27.0%
SOV,0.5,0.4,0.1,25.6%
HOV2,0.5,0.5,0.0,7.7%
HOV3+,0.5,0.5,-0.0,-1.1%
Transit Walk Access,0.4,0.3,0.1,21.0%
Transit Auto Access,0.3,0.4,-0.1,-30.5%
School Bus,0.2,0.1,0.1,37.0%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.3,0.3,0.0,1.4%
Bike,0.4,0.1,0.4,576.6%
SOV,0.3,0.2,0.0,5.6%
HOV2,0.4,0.2,0.1,56.6%
HOV3+,0.4,0.2,0.2,132.3%
Transit Walk Access,0.2,0.3,-0.1,-33.9%
Transit Auto Access,0.1,0.0,0.0,76.8%
School Bus,0.0,0.0,0.0,"1,017.8%"


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.2,-0.0,-12.6%
Bike,0.2,0.2,0.0,12.4%
SOV,0.1,0.1,0.0,6.2%
HOV2,0.4,0.4,-0.0,-2.9%
HOV3+,0.6,0.5,0.0,7.3%
Transit Walk Access,0.1,0.0,0.0,163.2%
Transit Auto Access,0.0,0.0,0.0,nan%
School Bus,0.1,0.1,-0.0,-24.8%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.1,0.1,0.0,34.6%
Bike,0.3,0.2,0.1,46.1%
SOV,0.1,0.1,0.0,82.3%
HOV2,0.1,0.1,-0.0,-22.1%
HOV3+,0.2,0.3,-0.1,-37.1%
Transit Walk Access,0.3,0.1,0.2,136.5%
Transit Auto Access,0.0,0.0,0.0,nan%
School Bus,1.0,1.0,0.0,4.5%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.2,0.0,1.6%
Bike,0.1,0.1,-0.0,-24.9%
SOV,0.1,0.1,0.0,10.9%
HOV2,0.2,0.2,-0.0,-19.2%
HOV3+,0.1,0.2,-0.0,-26.1%
Transit Walk Access,0.1,0.0,0.1,192.9%
Transit Auto Access,0.1,0.0,0.0,87.6%
School Bus,0.0,0.0,0.0,"2,059.1%"


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.1,0.1,92.2%
Bike,0.1,0.2,-0.0,-16.0%
SOV,0.2,0.1,0.0,18.5%
HOV2,0.2,0.1,0.0,5.6%
HOV3+,0.1,0.1,0.0,29.1%
Transit Walk Access,0.2,0.2,-0.0,-5.7%
Transit Auto Access,0.0,0.0,0.0,18.8%
School Bus,0.0,0.1,-0.1,-76.5%


In [17]:
tour_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.8,0.9,-0.1,-15.6%
Bike,0.9,0.5,0.3,68.5%
SOV,0.6,0.5,0.2,40.4%
HOV2,0.6,0.5,0.1,22.7%
HOV3+,0.6,0.7,-0.2,-25.8%
Transit Walk Access,0.6,0.4,0.2,51.0%
Transit Auto Access,0.4,0.7,-0.4,-50.3%
School Bus,0.2,0.3,-0.0,-9.9%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.5,0.4,0.0,6.7%
Bike,0.2,0.2,0.1,30.5%
SOV,0.6,0.7,-0.1,-15.3%
HOV2,0.3,0.2,0.1,18.8%
HOV3+,0.3,0.2,0.1,20.6%
Transit Walk Access,0.4,0.3,0.1,57.5%
Transit Auto Access,1.1,0.3,0.8,282.0%
School Bus,0.0,0.0,0.0,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.1,0.1,-0.0,-0.4%
Bike,0.2,0.5,-0.3,-59.0%
SOV,0.1,0.0,0.1,433.1%
HOV2,0.1,0.2,-0.0,-28.0%
HOV3+,0.2,0.4,-0.1,-41.2%
Transit Walk Access,0.2,0.4,-0.2,-41.4%
Transit Auto Access,0.0,0.0,0.0,nan%
School Bus,1.0,1.1,-0.1,-9.7%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.3,-0.1,-44.3%
Bike,0.3,0.6,-0.3,-53.6%
SOV,0.1,0.1,0.0,51.1%
HOV2,0.4,0.5,-0.1,-11.0%
HOV3+,0.5,0.5,0.1,11.1%
Transit Walk Access,0.1,0.0,0.1,"3,884.2%"
Transit Auto Access,0.0,0.0,0.0,nan%
School Bus,0.1,0.0,0.1,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.1,0.1,86.5%
Bike,0.2,0.0,0.2,nan%
SOV,0.2,0.2,0.1,33.0%
HOV2,0.2,0.1,0.1,67.1%
HOV3+,0.2,0.1,0.0,17.2%
Transit Walk Access,0.2,0.1,0.1,111.0%
Transit Auto Access,0.1,0.0,0.1,nan%
School Bus,0.0,0.0,0.0,639.4%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.2,-0.0,-13.8%
Bike,0.1,0.0,0.1,"3,740.6%"
SOV,0.2,0.1,0.1,47.9%
HOV2,0.2,0.1,0.1,56.7%
HOV3+,0.1,0.1,0.1,67.9%
Transit Walk Access,0.2,0.0,0.1,360.1%
Transit Auto Access,0.1,0.0,0.1,nan%
School Bus,0.0,0.0,0.0,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.1,0.0,0.1,"1,058.5%"
Bike,0.2,0.0,0.2,nan%
SOV,0.2,0.2,0.0,11.3%
HOV2,0.1,0.1,0.1,50.8%
HOV3+,0.1,0.0,0.1,168.3%
Transit Walk Access,0.2,0.1,0.1,264.4%
Transit Auto Access,0.0,0.0,0.0,nan%
School Bus,0.0,0.0,0.0,nan%


## Tour Share by Purpose

In [18]:
def pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Purpose
    tour_1_total = get_total(data_daysim['Tour']['toexpfac'])
    tour_2_total = get_total(data_survey['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'pdpurp', 'Tour Purpose')
    ptbp = ptbp.loc[pdpurp_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Purpose',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Tours by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Tours', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [19]:
pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Purpose,,,,
Work,26.9%,27.5%,-0.6%,-2.1%
School,9.8%,9.7%,0.2%,1.6%
Escort,11.1%,10.8%,0.3%,3.2%
Personal Business,7.0%,6.8%,0.2%,2.6%
Shop,12.9%,12.9%,0.0%,0.1%
Meal,7.2%,7.0%,0.2%,3.4%
Social,25.0%,25.3%,-0.3%,-1.3%


In [20]:
pc_tour_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Purpose,,,,
Work,1.9%,1.8%,0.1%,7.5%
School,0.9%,0.8%,0.1%,7.1%
Escort,0.8%,0.8%,0.1%,12.1%
Personal Business,0.5%,0.3%,0.2%,81.6%
Shop,0.8%,0.5%,0.3%,55.4%
Meal,0.7%,0.4%,0.2%,53.9%
Social,2.2%,2.0%,0.2%,12.2%


## Tour Share by Mode

In [21]:
def pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Mode
    tour_1_total = get_total(data_daysim['Tour']['toexpfac'])
    tour_2_total = get_total(data_survey['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'tmodetp', 'Tour Mode')
    ptbp = ptbp.loc[mode_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Mode',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Tour Share by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Tour Share', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [22]:
pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Mode,,,,
Walk,10.8%,10.9%,-0.1%,-1.1%
Bike,0.6%,1.2%,-0.7%,-54.5%
SOV,38.6%,37.3%,1.4%,3.6%
HOV2,22.3%,22.8%,-0.5%,-2.0%
HOV3+,21.6%,19.5%,2.1%,10.8%
Transit Walk Access,3.4%,4.3%,-0.8%,-19.9%
Transit Auto Access,0.2%,0.4%,-0.2%,-48.2%
School Bus,2.4%,2.5%,-0.1%,-2.8%


In [23]:
pc_tour_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Mode,,,,
Walk,0.6%,0.8%,-0.2%,-28.3%
Bike,0.0%,0.1%,-0.0%,-42.8%
SOV,3.0%,2.2%,0.8%,38.5%
HOV2,1.9%,1.6%,0.3%,17.3%
HOV3+,1.6%,1.1%,0.5%,47.8%
Transit Walk Access,0.5%,0.3%,0.2%,64.6%
Transit Auto Access,0.0%,0.2%,-0.1%,-86.1%
School Bus,0.2%,0.2%,0.0%,1.9%


## Tour Distance by Purpose

In [24]:
def tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl = atl.loc[pdpurp_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.1f}',
        f'Average Tour Length ({name2})': '{:,.1f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [25]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,11.5,11.8,-0.3,-2.6%
School,4.8,4.6,0.3,5.8%
Escort,7.4,7.1,0.3,4.1%
Personal Business,6.7,8.0,-1.3,-15.9%
Shop,4.7,4.7,0.0,0.6%
Meal,3.6,3.8,-0.1,-4.1%
Social,5.8,6.9,-1.1,-16.2%


In [26]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,9.2,7.0,2.2,31.0%
School,3.9,4.3,-0.3,-8.2%
Escort,6.6,11.2,-4.7,-41.6%
Personal Business,7.1,5.7,1.4,24.9%
Shop,2.7,4.0,-1.3,-32.2%
Meal,2.9,5.3,-2.4,-45.9%
Social,4.9,8.0,-3.1,-39.1%


## Tour Distance by Mode

In [27]:
def tour_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    #Average Distance by Tour Mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl = atl.loc[mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.1f}',
        f'Average Tour Length ({name2})': '{:,.1f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [28]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,11.5,11.8,-0.3,-2.6%
School,4.8,4.6,0.3,5.8%
Escort,7.4,7.1,0.3,4.1%
Personal Business,6.7,8.0,-1.3,-15.9%
Shop,4.7,4.7,0.0,0.6%
Meal,3.6,3.8,-0.1,-4.1%
Social,5.8,6.9,-1.1,-16.2%


In [29]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,9.2,7.0,2.2,31.0%
School,3.9,4.3,-0.3,-8.2%
Escort,6.6,11.2,-4.7,-41.6%
Personal Business,7.1,5.7,1.4,24.9%
Shop,2.7,4.0,-1.3,-32.2%
Meal,2.9,5.3,-2.4,-45.9%
Social,4.9,8.0,-3.1,-39.1%


## Tour Travel Time by Purpose

In [30]:
def tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[pdpurp_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.1f}',
        f'Average Tour Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [31]:
tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Purpose,,,,
Work,28.8,25.3,3.5,13.7%
School,15.2,13.4,1.9,13.9%
Escort,21.2,16.1,5.1,31.4%
Personal Business,20.1,37.6,-17.5,-46.5%
Shop,16.5,14.4,2.2,15.1%
Meal,13.6,16.7,-3.1,-18.8%
Social,17.5,17.4,0.1,0.6%


In [32]:
tours_tt_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Purpose,,,,
Work,18.9,17.5,1.4,8.2%
School,8.8,12.1,-3.2,-26.8%
Escort,13.7,23.2,-9.5,-40.9%
Personal Business,14.9,14.1,0.9,6.2%
Shop,6.8,12.2,-5.4,-44.1%
Meal,7.2,13.5,-6.3,-46.9%
Social,10.7,20.6,-9.9,-48.1%


## Tour Travel Time by Mode

In [33]:
def tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[mode_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.1f}',
        f'Average Tour Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [34]:
tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Mode,,,,
Walk,3.6,19.1,-15.4,-81.0%
Bike,9.0,11.9,-2.9,-24.1%
SOV,24.4,19.6,4.8,24.8%
HOV2,22.1,17.7,4.3,24.4%
HOV3+,22.2,17.8,4.4,24.7%
Transit Walk Access,26.9,67.9,-41.0,-60.4%
Transit Auto Access,35.5,37.9,-2.5,-6.5%
School Bus,13.0,10.5,2.4,23.0%


In [35]:
tours_tt_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Mode,,,,
Walk,2.7,12.7,-9.9,-78.6%
Bike,7.2,10.2,-3.0,-29.0%
SOV,14.4,14.3,0.1,0.5%
HOV2,13.5,23.7,-10.2,-43.1%
HOV3+,13.2,16.4,-3.2,-19.4%
Transit Walk Access,13.8,22.7,-8.9,-39.4%
Transit Auto Access,15.0,37.5,-22.5,-59.9%
School Bus,6.9,8.7,-1.8,-20.2%


## Tours by District

In [36]:
##Tours per Person by Purpose and Person Type/Number of Stops
data1=data_daysim
data2=data_survey
name1 = 'DaysimOutputs'
name2 = f'{survey_year}Survey' 
# calculate the percentage of each number of stops by purpose
data1['Household'] = data1['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data1['Tour'] = data1['Tour'].merge(data1['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
data2['Household'] = data2['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data2['Tour_cloned'] = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
todist1 = data1['Tour'].groupby(by='DistrictFlowName')['toexpfac'].sum()
todist2 = data2['Tour_cloned'].groupby(by='DistrictFlowName')['toexpfac'].sum()
df_compare = pd.concat([todist1, todist2], axis=1)
df_compare.columns = [name1, name2]
df_compare['Difference'] = df_compare[name1] - df_compare[name2]
df_compare['% Difference'] = (df_compare['Difference'] / df_compare[name2]) * 100
display(df_compare.loc[district_flow_name.values()].style.format({
    name1: '{:,.0f}',
    name2: '{:,.0f}',
    'Difference': '{:,.0f}',
    '% Difference': '{:,.1f}%'
}))

,DaysimOutputs,2023Survey,Difference,% Difference
DistrictFlowName,,,,
Bellevue (excluding downtown),"195,235","156,198","39,037",25.0%
Bellevue Downtown,"27,127","7,684","19,443",253.0%
Kirkland,"134,487","61,197","73,290",119.8%
Redmond,"105,592","107,608","-2,016",-1.9%
Seattle (excluding Seattle downtown),"1,022,144","799,965","222,179",27.8%
Seattle downtown,"149,741","73,367","76,374",104.1%
Rest,"4,243,748","3,003,645","1,240,103",41.3%


In [37]:
def tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Tour Purpose',
                        gp1_list=[], gp2_list=[]):
    tour_by_district_purpose1 = data1['Tour'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    tour_by_district_purpose2 = data2['Tour_cloned'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    tour_by_district_purpose1 = tour_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose2 = tour_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose1.columns.name = gp2_label
    tour_by_district_purpose2.columns.name = gp2_label
    tour_by_district_purpose1.index.name = gp1_label
    tour_by_district_purpose2.index.name = gp1_label
    display(tour_by_district_purpose1.style.format('{:,.0f}').set_caption("DaysimOutputs"))
    display(tour_by_district_purpose2.style.format('{:,.0f}').set_caption(f"{survey_year}Survey"))
    percent_diff = (tour_by_district_purpose1 - tour_by_district_purpose2) / tour_by_district_purpose2 * 100
    display(percent_diff.style.format('{:,.1f}%').set_caption("Percentage Difference (DaysimOutputs - Survey)"))

In [38]:
def tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Tour Purpose',
                        gp1_list=[], gp2_list=[]):
    tour_by_district_purpose1 = data1['Tour'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    tour_by_district_purpose2 = data2['Tour_cloned'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    tour_by_district_purpose1 = tour_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose2 = tour_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose1.columns.name = gp2_label
    tour_by_district_purpose2.columns.name = gp2_label
    tour_by_district_purpose1.index.name = gp1_label
    tour_by_district_purpose2.index.name = gp1_label
    tour_by_district_purpose1 = tour_by_district_purpose1.div(tour_by_district_purpose1.sum(axis=0), axis=1) * 100
    tour_by_district_purpose2 = tour_by_district_purpose2.div(tour_by_district_purpose2.sum(axis=0), axis=1) * 100
    display(tour_by_district_purpose1.style.format('{:,.1f}%').set_caption("DaysimOutputs"))
    display(tour_by_district_purpose2.style.format('{:,.1f}%').set_caption(f"{survey_year}Survey"))

## Tours by District by Purpose

In [39]:
tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='pdpurp', gp2='DistrictFlowName', 
                        gp1_label='Tour Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,"45,380","6,209","35,164","27,676","323,372","57,647","1,086,574"
School,"24,632","1,784","13,664","10,750","99,668","5,671","422,290"
Escort,"21,941","2,301","14,385","11,279","85,601","7,344","511,911"
Personal Business,"12,397","2,362","8,853","6,945","65,631","10,243","306,790"
Shop,"19,282","3,283","13,397","10,431","97,075","14,795","602,128"
Meal,"16,508","3,013","11,561","9,169","86,306","15,271","280,479"
Social,"55,095","8,175","37,463","29,342","264,491","38,770","1,033,576"


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,"42,306","1,628","21,347","26,193","216,041","18,412","840,701"
School,"15,741",460,"6,663","17,917","89,718","2,193","326,254"
Escort,"19,935",176,"11,294","6,841","100,471","5,755","319,216"
Personal Business,"8,917",825,"1,098","3,616","46,592","7,087","232,686"
Shop,"13,154","1,140","6,288","5,071","80,216","11,157","405,194"
Meal,"8,593","1,283","2,940","9,653","71,605","8,958","160,869"
Social,"47,552","2,172","11,567","38,317","195,321","19,806","718,726"


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,7.3%,281.4%,64.7%,5.7%,49.7%,213.1%,29.2%
School,56.5%,288.1%,105.1%,-40.0%,11.1%,158.5%,29.4%
Escort,10.1%,"1,210.2%",27.4%,64.9%,-14.8%,27.6%,60.4%
Personal Business,39.0%,186.3%,706.2%,92.1%,40.9%,44.5%,31.8%
Shop,46.6%,187.9%,113.1%,105.7%,21.0%,32.6%,48.6%
Meal,92.1%,134.8%,293.2%,-5.0%,20.5%,70.5%,74.4%
Social,15.9%,276.3%,223.9%,-23.4%,35.4%,95.8%,43.8%


In [40]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='pdpurp', gp2='DistrictFlowName', 
                        gp1_label='Tour Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,23.2%,22.9%,26.1%,26.2%,31.6%,38.5%,25.6%
School,12.6%,6.6%,10.2%,10.2%,9.8%,3.8%,10.0%
Escort,11.2%,8.5%,10.7%,10.7%,8.4%,4.9%,12.1%
Personal Business,6.3%,8.7%,6.6%,6.6%,6.4%,6.8%,7.2%
Shop,9.9%,12.1%,10.0%,9.9%,9.5%,9.9%,14.2%
Meal,8.5%,11.1%,8.6%,8.7%,8.4%,10.2%,6.6%
Social,28.2%,30.1%,27.9%,27.8%,25.9%,25.9%,24.4%


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,27.1%,21.2%,34.9%,24.3%,27.0%,25.1%,28.0%
School,10.1%,6.0%,10.9%,16.7%,11.2%,3.0%,10.9%
Escort,12.8%,2.3%,18.5%,6.4%,12.6%,7.8%,10.6%
Personal Business,5.7%,10.7%,1.8%,3.4%,5.8%,9.7%,7.7%
Shop,8.4%,14.8%,10.3%,4.7%,10.0%,15.2%,13.5%
Meal,5.5%,16.7%,4.8%,9.0%,9.0%,12.2%,5.4%
Social,30.4%,28.3%,18.9%,35.6%,24.4%,27.0%,23.9%


## Tours by District by Mode

In [41]:
tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='tmodetp', 
                    gp1_label='DistrictFlowName', gp2_label='Tour Mode',
                    gp1_list=district_flow_name.values(), gp2_list=mode_cat.values())

Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),"10,317",828,"72,625","47,454","42,486","14,191",740,"6,594"
Bellevue Downtown,"8,800",484,"8,088","4,586","1,990","2,762",67,350
Kirkland,"9,406",486,"53,236","33,405","26,408","7,993",277,"3,276"
Redmond,"7,340",555,"42,267","25,061","20,909","6,495",341,"2,624"
Seattle (excluding Seattle downtown),"189,246","7,990","387,431","188,125","151,150","75,360","2,413","20,429"
Seattle downtown,"83,871","1,775","34,245","12,402","5,726","11,059",328,335
Rest,"326,581","21,321","1,673,106","999,200","1,023,275","84,016","6,993","109,256"


Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),"24,859","1,722","40,509","52,129","28,415","4,405",nan,"1,353"
Bellevue Downtown,"1,909",187,"2,728","2,175",208,425,nan,nan
Kirkland,"9,449",nan,"25,981","10,864","8,783","3,146",421,"1,500"
Redmond,"6,735","1,627","40,100","15,781","15,944","8,441","8,364","7,973"
Seattle (excluding Seattle downtown),"143,556","29,563","252,773","206,703","103,882","50,194",385,"6,909"
Seattle downtown,"30,339","2,951","14,023","12,858","3,386","7,078",985,27
Rest,"191,968","14,026","1,157,926","673,840","720,968","115,053","4,303","88,544"


Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),-58.5%,-51.9%,79.3%,-9.0%,49.5%,222.2%,nan%,387.2%
Bellevue Downtown,361.0%,158.6%,196.4%,110.9%,856.7%,549.7%,nan%,nan%
Kirkland,-0.5%,nan%,104.9%,207.5%,200.7%,154.1%,-34.1%,118.4%
Redmond,9.0%,-65.9%,5.4%,58.8%,31.1%,-23.0%,-95.9%,-67.1%
Seattle (excluding Seattle downtown),31.8%,-73.0%,53.3%,-9.0%,45.5%,50.1%,526.0%,195.7%
Seattle downtown,176.4%,-39.9%,144.2%,-3.5%,69.1%,56.2%,-66.7%,"1,118.6%"
Rest,70.1%,52.0%,44.5%,48.3%,41.9%,-27.0%,62.5%,23.4%


In [42]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='tmodetp', 
                    gp1_label='DistrictFlowName', gp2_label='Tour Mode',
                    gp1_list=district_flow_name.values(), gp2_list=mode_cat.values())

Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),1.6%,2.5%,3.2%,3.6%,3.3%,7.0%,6.6%,4.6%
Bellevue Downtown,1.4%,1.4%,0.4%,0.4%,0.2%,1.4%,0.6%,0.2%
Kirkland,1.5%,1.5%,2.3%,2.5%,2.1%,4.0%,2.5%,2.3%
Redmond,1.2%,1.7%,1.9%,1.9%,1.6%,3.2%,3.1%,1.8%
Seattle (excluding Seattle downtown),29.8%,23.9%,17.1%,14.4%,11.9%,37.3%,21.6%,14.3%
Seattle downtown,13.2%,5.3%,1.5%,0.9%,0.5%,5.5%,2.9%,0.2%
Rest,51.4%,63.8%,73.7%,76.3%,80.4%,41.6%,62.7%,76.5%


Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),6.1%,3.4%,2.6%,5.4%,3.2%,2.3%,nan%,1.3%
Bellevue Downtown,0.5%,0.4%,0.2%,0.2%,0.0%,0.2%,nan%,nan%
Kirkland,2.3%,nan%,1.7%,1.1%,1.0%,1.7%,2.9%,1.4%
Redmond,1.6%,3.2%,2.6%,1.6%,1.8%,4.5%,57.9%,7.5%
Seattle (excluding Seattle downtown),35.1%,59.0%,16.5%,21.2%,11.8%,26.6%,2.7%,6.5%
Seattle downtown,7.4%,5.9%,0.9%,1.3%,0.4%,3.8%,6.8%,0.0%
Rest,47.0%,28.0%,75.5%,69.2%,81.8%,61.0%,29.8%,83.3%


## Tours by Purpose by Auto Sufficiency

In [43]:
def classify_auto_sufficiency(row):
    if row['hhvehs'] == 0:
        return '0 veh'
    elif row['hhvehs'] < row['driver']:
        return 'autos < drivers'
    elif row['hhvehs'] == row['driver']:
        return 'autos = drivers'
    else:
        return 'autos > drivers'

data1['Trip']['driver'] = (data1['Trip']['dorp'] == 'Driver').astype(int)
data2['Trip_cloned']['driver'] = (data2['Trip_cloned']['dorp'] == 'Driver').astype(int)

_driver1 = data1['Trip'][['hhno', 'pno', 'driver']].groupby(by=['hhno', 'pno'])['driver'].sum().reset_index()
_driver1['driver'] = (_driver1['driver'] > 0).astype(int)
driver1 = _driver1.groupby(by='hhno')['driver'].sum()

_driver2 = data2['Trip_cloned'][['hhno', 'pno', 'driver']].groupby(by=['hhno', 'pno'])['driver'].sum().reset_index()
_driver2['driver'] = (_driver2['driver'] > 0).astype(int)
driver2 = _driver2.groupby(by='hhno')['driver'].sum()

data1['Tour'] = data1['Tour'].merge(driver1, on='hhno', how='left')
data2['Tour_cloned'] = data2['Tour_cloned'].merge(driver2, on='hhno', how='left')

data1['Tour']['auto_sufficiency'] = data1['Tour'][['hhvehs', 'driver']].apply(classify_auto_sufficiency, axis=1)
data2['Tour_cloned']['auto_sufficiency'] = data2['Tour_cloned'][['hhvehs', 'driver']].apply(classify_auto_sufficiency, axis=1)

In [44]:
autosuf_labels = ['0 veh', 'autos < drivers', 'autos = drivers', 'autos > drivers']

tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='auto_sufficiency', 
                    gp1_label='Tour Purpose', gp2_label='Auto Sufficiency',
                    gp1_list=pdpurp_cat.values(), gp2_list=autosuf_labels)

Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,"32,894","275,585","813,834","459,709"
School,"22,833","131,997","256,393","167,236"
Escort,"24,207","140,271","309,804","180,480"
Personal Business,"37,757","64,983","206,470","104,011"
Shop,"70,104","125,940","354,775","209,572"
Meal,"19,194","82,770","205,351","114,992"
Social,"60,134","238,359","728,194","440,225"


Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,"26,609","32,601","515,906","814,078"
School,"9,931","11,824","174,176","293,231"
Escort,"1,508","7,490","204,598","331,320"
Personal Business,"30,306","10,406","129,196","176,184"
Shop,"78,908","16,338","170,704","386,900"
Meal,"10,058","16,156","87,616","237,257"
Social,"20,065","19,862","342,370","895,123"


Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,23.6%,745.3%,57.7%,-43.5%
School,129.9%,"1,016.3%",47.2%,-43.0%
Escort,"1,505.2%","1,772.8%",51.4%,-45.5%
Personal Business,24.6%,524.5%,59.8%,-41.0%
Shop,-11.2%,670.8%,107.8%,-45.8%
Meal,90.8%,412.3%,134.4%,-51.5%
Social,199.7%,"1,100.1%",112.7%,-50.8%


In [45]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='auto_sufficiency', 
                    gp1_label='Tour Purpose', gp2_label='Auto Sufficiency',
                    gp1_list=pdpurp_cat.values(), gp2_list=autosuf_labels)

Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,12.3%,26.0%,28.3%,27.4%
School,8.5%,12.5%,8.9%,10.0%
Escort,9.1%,13.2%,10.8%,10.8%
Personal Business,14.1%,6.1%,7.2%,6.2%
Shop,26.2%,11.9%,12.3%,12.5%
Meal,7.2%,7.8%,7.1%,6.9%
Social,22.5%,22.5%,25.3%,26.3%


Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,15.0%,28.4%,31.8%,26.0%
School,5.6%,10.3%,10.7%,9.4%
Escort,0.9%,6.5%,12.6%,10.6%
Personal Business,17.1%,9.1%,8.0%,5.6%
Shop,44.5%,14.2%,10.5%,12.3%
Meal,5.7%,14.1%,5.4%,7.6%
Social,11.3%,17.3%,21.1%,28.6%


## Tours by Purpose by Income Level

In [46]:
income_bins = [0, 25000, 45000, 75000, float('inf')]
income_labels = ['Less than 25,000', '$25,000-$44,999', '$45,000-74,999', 'More than $75,000']
data1['Tour']['income_group'] = pd.cut(data1['Tour']['hhincome'], bins=income_bins, labels=income_labels, right=False)
data2['Tour_cloned']['income_group'] = pd.cut(data2['Tour_cloned']['hhincome'], bins=income_bins, labels=income_labels, right=False)

tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='income_group', 
                    gp1_label='Tour Purpose', gp2_label='Income Level',
                    gp1_list=pdpurp_cat.values(), gp2_list=income_labels)

Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,"92,182","180,260","324,366","984,904"
School,"74,136","67,925","109,374","326,812"
Escort,"75,982","77,617","135,642","365,338"
Personal Business,"86,724","73,573","82,136","170,627"
Shop,"139,965","113,180","158,251","348,716"
Meal,"40,925","55,209","85,246","240,775"
Social,"153,145","192,866","296,786","823,652"


Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,"34,915","90,047","155,619","769,885"
School,"22,898","10,638","79,640","307,323"
Escort,"5,669","33,867","85,250","294,494"
Personal Business,"41,020","35,946","25,549","159,770"
Shop,"57,193","97,456","65,293","254,464"
Meal,"32,658","22,226","25,663","159,646"
Social,"70,500","79,700","146,545","635,093"


Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,164.0%,100.2%,108.4%,27.9%
School,223.8%,538.5%,37.3%,6.3%
Escort,"1,240.4%",129.2%,59.1%,24.1%
Personal Business,111.4%,104.7%,221.5%,6.8%
Shop,144.7%,16.1%,142.4%,37.0%
Meal,25.3%,148.4%,232.2%,50.8%
Social,117.2%,142.0%,102.5%,29.7%


In [47]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='income_group', 
                    gp1_label='Tour Purpose', gp2_label='Income Level',
                    gp1_list=pdpurp_cat.values(), gp2_list=income_labels)

Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,13.9%,23.7%,27.2%,30.2%
School,11.2%,8.9%,9.2%,10.0%
Escort,11.5%,10.2%,11.4%,11.2%
Personal Business,13.1%,9.7%,6.9%,5.2%
Shop,21.1%,14.9%,13.3%,10.7%
Meal,6.2%,7.3%,7.2%,7.4%
Social,23.1%,25.4%,24.9%,25.3%


Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,13.2%,24.3%,26.7%,29.8%
School,8.6%,2.9%,13.6%,11.9%
Escort,2.1%,9.2%,14.6%,11.4%
Personal Business,15.5%,9.7%,4.4%,6.2%
Shop,21.6%,26.3%,11.2%,9.9%
Meal,12.3%,6.0%,4.4%,6.2%
Social,26.6%,21.5%,25.1%,24.6%
